# 🔧 Notebook 2 — Data Preprocessing & Augmentation
## Aerial Bird vs Drone Classification

**Objectives:**
- Build train/val/test image generators
- Apply and visualize augmentation techniques
- Inspect preprocessed batches

In [ ]:
import os, sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path(os.getcwd()).parent
sys.path.insert(0, str(ROOT))

from src.config import TRAIN_DIR, VALID_DIR, TEST_DIR, BATCH_SIZE, IMG_SIZE
from src.data_preprocessing import (
    get_all_generators, visualize_augmentation, dataset_summary
)
from src.utils import set_seed

set_seed()
dataset_summary()
print('✅ Setup complete')

## 1. Create Data Generators

In [ ]:
train_gen, valid_gen, test_gen = get_all_generators(batch_size=BATCH_SIZE)

print('\nClass Indices:', train_gen.class_indices)
print('Train batches:', len(train_gen))
print('Valid batches:', len(valid_gen))
print('Test  batches:', len(test_gen))

## 2. Inspect a Preprocessed Batch

In [ ]:
# Fetch one batch
images, labels = next(train_gen)
print(f'Batch shape : {images.shape}')
print(f'Labels      : {labels[:8]}')
print(f'Pixel range : [{images.min():.3f}, {images.max():.3f}]')
print(f'Pixel mean  : {images.mean():.4f}')

# Plot batch
n_show = min(8, len(images))
fig, axes = plt.subplots(2, n_show//2, figsize=(16, 5))
axes = axes.ravel()
class_names = {v: k for k, v in train_gen.class_indices.items()}

for i in range(n_show):
    axes[i].imshow(images[i])
    label = class_names[int(labels[i])]
    color = '#4CAF50' if label == 'bird' else '#f44336'
    axes[i].set_title(label.upper(), color=color, fontsize=10, fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Preprocessed Training Batch', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(ROOT / 'results' / 'plots' / 'preprocessed_batch.png'), dpi=150)
plt.show()

## 3. Augmentation Visualization

In [ ]:
# Show augmentation on a sample image from each class
for cls in ['bird', 'drone']:
    cls_dir = os.path.join(TRAIN_DIR, cls)
    sample  = os.listdir(cls_dir)[0]
    img_path = os.path.join(cls_dir, sample)
    print(f'\nAugmenting: {cls.upper()}')
    visualize_augmentation(
        img_path, n_augments=7,
        save_path=str(ROOT / 'results' / 'plots' / f'augmentation_{cls}.png')
    )

## 4. Augmentation Parameters Summary

In [ ]:
from src.config import (
    ROTATION_RANGE, WIDTH_SHIFT_RANGE, HEIGHT_SHIFT_RANGE,
    SHEAR_RANGE, ZOOM_RANGE, HORIZONTAL_FLIP, BRIGHTNESS_RANGE
)

print('='*50)
print('  AUGMENTATION CONFIG')
print('='*50)
params = {
    'Rotation Range':   f'{ROTATION_RANGE}°',
    'Width Shift':      f'{WIDTH_SHIFT_RANGE*100:.0f}%',
    'Height Shift':     f'{HEIGHT_SHIFT_RANGE*100:.0f}%',
    'Shear Range':      f'{SHEAR_RANGE}',
    'Zoom Range':       f'{ZOOM_RANGE}',
    'Horizontal Flip':  str(HORIZONTAL_FLIP),
    'Brightness Range': str(BRIGHTNESS_RANGE),
    'Rescale':          '1/255 → [0,1]',
    'Fill Mode':        'nearest',
}
for k, v in params.items():
    print(f'  {k:<20}: {v}')
print('='*50)
print('\n✅ Augmentations only applied to TRAINING set')
print('   Validation & Test sets: rescale only')